<a href="https://colab.research.google.com/github/latikasekhri6577/Langchain-LangGraph-Course/blob/main/Langchain_RAG_Course.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. Install Packages**

Installs the LangChain libraries into Colab. You must run it again after every session restart.

In [2]:
!pip install -q langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 8.4 MB/s eta 0:00:00


In [12]:
!pip install -q langchain langchain-google-genai pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 15.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.59.1 which is incompatible.


**2.Load the Key and Create the Model**

Reads your secret key and creates the model object, which is your connection to the LLM. Swapping providers only changes this cell.

In [14]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

**3. Call the LLM**

model.invoke("...") sends text to the model and returns a message object. .content is the answer text.

In [15]:
response = model.invoke("Explain what an API is in 2 lines.")
print(response.content)

An API (Application Programming Interface) is a set of definitions and protocols that allows different software applications to communicate. It acts as an intermediary, enabling programs to request services or exchange data with each other.


**4. Prompt Templates**

A reusable prompt with blanks like {topic} that you fill in each time. The system message sets the model's role, and the human message holds the question.

In [16]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly teacher who explains things simply."),
    ("human", "Explain {topic} to a {level} in 3 lines."),
])

messages = prompt.invoke({"topic": "recursion", "level": "beginner"})
print(messages)

response = model.invoke(messages)
print(response.content)

messages=[SystemMessage(content='You are a friendly teacher who explains things simply.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain recursion to a beginner in 3 lines.', additional_kwargs={}, response_metadata={})]
Imagine a function that solves a problem by calling itself.
It keeps breaking the problem into smaller, identical pieces until it reaches a simple base case.
Crucially, it needs a "stop" condition, or it will repeat forever!


**5. Parsing the output**

StrOutputParser pulls the plain text out of the model's message object. This gives you a clean string to use in later steps.

In [17]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

text = parser.invoke(response)
print(text)
print(type(text))

Imagine a function that solves a problem by calling itself.
It keeps breaking the problem into smaller, identical pieces until it reaches a simple base case.
Crucially, it needs a "stop" condition, or it will repeat forever!
<class 'langchain_core.messages.base.TextAccessor'>


**6. Simple chain**

prompt | model | parser joins the three pieces with a pipe, so one invoke call runs all of them in order.

In [18]:
chain = prompt | model | parser

answer = chain.invoke({"topic": "recursion", "level": "beginner"})
print(answer)

Imagine a function that solves a problem by calling itself.
It keeps breaking the problem into smaller, identical pieces until it reaches a simple, solvable step.
Crucially, it needs a "stop condition" to know when to finish and avoid repeating forever!


**7. Structured Output**

You define a Pydantic class and the model returns an object of that shape. You can then read result.name directly instead of parsing a paragraph.

In [19]:
from pydantic import BaseModel, Field
from typing import List

class Concept(BaseModel):
    name: str = Field(description="Name of the concept")
    definition: str = Field(description="One-line definition")
    examples: List[str] = Field(description="Two short examples")
    difficulty: str = Field(description="easy, medium or hard")

structured_model = model.with_structured_output(Concept)

result = structured_model.invoke("Tell me about recursion.")
print(result)
print(result.name)
print(result.examples[0])

name='Recursion' definition='A programming technique where a function calls itself to solve a problem by breaking it down into smaller, similar subproblems.' examples=['Calculating the factorial of a number (e.g., factorial(n) = n * factorial(n-1)).', 'Traversing a tree data structure (e.g., visiting a node, then recursively visiting its children).'] difficulty='medium'
Recursion
Calculating the factorial of a number (e.g., factorial(n) = n * factorial(n-1)).
